### packege we will use

In [ ]:
[workspace]
resolver = "2"
members = [
    "module-01-intro",
    "module-02-routing",
    "module-03-extractors",
    "module-04-responses",
    "module-05-state",
    "module-06-middleware",
    "module-07-errors",
    "module-08-database",
    "module-09-auth",
    "module-10-advanced",
    "module-11-testing",
    "module-12-production",
]

# Shared dependencies across all modules
[workspace.dependencies]
# Core Axum Framework
axum = "0.8.8"
axum-extra = { version = "0.10", features = ["typed-header", "query", "multipart"] }

# Async Runtime
tokio = { version = "1.42", features = ["full"] }

# Serialization
serde = { version = "1.0", features = ["derive"] }
serde_json = "1.0"

# Tower Ecosystem
tower = { version = "0.5", features = ["full"] }
tower-http = { version = "0.6", features = ["cors", "compression-gzip", "timeout", "trace", "fs", "limit"] }

# Database
sqlx = { version = "0.8", features = ["runtime-tokio", "postgres", "uuid", "chrono"] }

# Authentication
jsonwebtoken = "9.3"
argon2 = "0.5"

# Error Handling
thiserror = "2.0"
anyhow = "1.0"

# Logging & Tracing
tracing = "0.1"
tracing-subscriber = { version = "0.3", features = ["env-filter", "json"] }

# Utilities
uuid = { version = "1.11", features = ["v4", "serde"] }
chrono = { version = "0.4", features = ["serde"] }
dotenvy = "0.15"
futures = "0.3"

# Testing
tower-service = "0.3"
http-body-util = "0.1"

In [ ]:
[package]
name = "axum-rust-api-handbook"
version = "0.1.0"
edition = "2024"

[dependencies]
anyhow = "1.0.104"
argon2 = "0.5.3"
axum = "0.8.9"
chrono = "0.4.45"
dotenvy = "0.15.7"
futures = "0.3.34"
http-body-util = "0.1.5"
jsonwebtoken = "11.0.0"
serde = { version = "1.0.229", features = ["derive"] }
serde_json = "1.0.151"
sqlx = "0.9.0"
thiserror = "2.0.20"
tokio = { version = "1.53.1", features = ["full"] }
tower = "0.5.3"
tower-http = "0.7.0"
tower-service = "0.3.3"
tracing = "0.1.44"
tracing-subscriber = "0.3.23"
uuid = "1.24.1"


**Error handling**
- `anyhow` — ergonomic error type for application code; lets you use `?` without defining custom error enums everywhere.
- `thiserror` — derive macro for defining your own structured error types (used more in libraries or when you need typed errors).

**Web framework / async runtime**
- `axum` — the HTTP web framework itself (routing, extractors, handlers), built on top of `tower` and `hyper`.
- `tokio` — the async runtime that powers everything (`full` pulls in all features: TCP, timers, macros, multi-threaded scheduler, etc.).
- `tower` — middleware/service abstraction (`Service`/`Layer` traits) that axum is built on; used for things like timeouts, rate limiting, retries.
- `tower-service` — just the core `Service` trait, a lighter dependency than full `tower`.
- `tower-http` — ready-made HTTP middleware (CORS, compression, tracing, request logging, static file serving) built on `tower`.
- `http-body-util` — utilities for working with the `http-body` crate's `Body` trait (e.g. collecting a streaming body into bytes).

**Database**
- `sqlx` — async SQL toolkit (Postgres/MySQL/SQLite) with compile-time checked queries.

**Auth / security**
- `argon2` — password hashing algorithm implementation (for storing user passwords securely).
- `jsonwebtoken` — encode/decode/verify JWTs (for auth tokens).

**Serialization**
- `serde` — the serialization/deserialization framework (`derive` feature gives you `#[derive(Serialize, Deserialize)]`).
- `serde_json` — JSON support for serde (parsing request/response bodies).

**Utilities**
- `chrono` — date/time handling (timestamps, formatting, durations).
- `uuid` — generate/parse UUIDs (commonly used for primary keys).
- `dotenvy` — loads environment variables from a `.env` file (maintained fork of the old `dotenv` crate).
- `futures` — extra combinators/utilities for working with `Future`/`Stream` beyond what std provides.

**Logging/observability**
- `tracing` — structured, async-aware logging/instrumentation (spans + events).
- `tracing-subscriber` — collects and formats `tracing` output (e.g. prints to stdout, filters by log level via `RUST_LOG`).



### important Basics

In [ ]:
//! # Module 01: Introduction to Axum
//!
//! This module covers the basics of setting up an Axum web server.
//! We'll learn about:
//! - Creating a basic server with `axum::serve`
//! - Defining route handlers
//! - Understanding the Router type
//! - Basic request/response flow

use axum::{
    routing::{get, post},
    Router,
};

// ============================================================================
// LESSON 1: Your First Handler
// ============================================================================

/// The simplest possible handler - just returns a string
/// 
/// In Axum, a handler is any async function that returns something
/// implementing `IntoResponse`. Strings automatically implement this!
async fn hello_world() -> &'static str {
    "Hello, World! 🦀"
}

/// Handlers can return different types
/// Here we return a String (owned) instead of &str (borrowed)
async fn hello_axum() -> String {
    format!("Welcome to Axum {}!", env!("CARGO_PKG_VERSION"))
}

// ============================================================================
// LESSON 2: The Health Check Pattern
// ============================================================================

/// A common pattern - health check endpoint for monitoring
/// 
/// This is essential for:
/// - Load balancers to check if your service is alive
/// - Kubernetes liveness/readiness probes
/// - Monitoring systems
async fn health_check() -> &'static str {
    "OK"
}

// ============================================================================
// LESSON 3: Multiple Response Types
// ============================================================================

/// You can return tuples for more control
/// (StatusCode, headers, body) or (StatusCode, body)
use axum::http::StatusCode;

async fn with_status() -> (StatusCode, &'static str) {
    (StatusCode::CREATED, "Resource created!")
}

/// Return different status codes based on conditions
async fn conditional_response() -> (StatusCode, &'static str) {
    let is_working = true;
    
    if is_working {
        (StatusCode::OK, "Everything is working!")
    } else {
        (StatusCode::SERVICE_UNAVAILABLE, "Service is down")
    }
}

// ============================================================================
// LESSON 4: POST Handlers (Preview of Extractors)
// ============================================================================

/// A simple POST handler that echoes back the body
/// We'll learn more about extractors in Module 03
async fn echo(body: String) -> String {
    format!("You sent: {}", body)
}

// ============================================================================
// MAIN: Building the Router and Serving
// ============================================================================

#[tokio::main]
async fn main() {
    // Build our application router
    // 
    // The Router is the core of Axum - it maps paths to handlers
    // You can chain multiple routes together using the builder pattern
    let app = Router::new()
        // Basic GET routes
        .route("/", get(hello_world))
        .route("/hello", get(hello_axum))
        .route("/health", get(health_check))
        
        // Routes with different status codes
        .route("/created", get(with_status))
        .route("/status", get(conditional_response))
        
        // POST route (we'll explore this more later)
        .route("/echo", post(echo));

    // Create a TCP listener
    // 
    // In Axum 0.8+, we use `axum::serve` instead of hyper::Server
    // This is the new, simplified way to run an Axum application
    let listener = tokio::net::TcpListener::bind("0.0.0.0:3000")
        .await
        .expect("Failed to bind to port 3000");

    println!("🚀 Module 01: Introduction to Axum");
    println!("   Server running on http://localhost:3000");
    println!();
    println!("📝 Try these endpoints:");
    println!("   GET  /        - Hello World");
    println!("   GET  /hello   - Welcome message");
    println!("   GET  /health  - Health check");
    println!("   GET  /created - Status code example");
    println!("   POST /echo    - Echo back your message");
    println!();
    println!("💡 Example: curl http://localhost:3000/hello");
    println!("💡 Example: curl -X POST -d 'Hello!' http://localhost:3000/echo");

    // Start serving requests
    // 
    // `axum::serve` is the new function in Axum 0.8+
    // It replaces the old hyper::Server approach
    axum::serve(listener, app)
        .await
        .expect("Server failed to start");
}

### Module 01: Introduction to Axum

Welcome to the first module! Here you'll learn the fundamentals of building web servers with Axum.

#### 🎯 What You'll Learn

- Setting up a basic Axum server
- Creating route handlers
- Using `axum::serve` (new in Axum 0.8+)
- Understanding the Router type
- Returning different response types

#### 🚀 Running This Module

```bash
cargo run
```

Server starts at: http://localhost:3000

#### 📝 Endpoints

| Method | Path | Description |
|--------|------|-------------|
| GET | `/` | Hello World |
| GET | `/hello` | Welcome message |
| GET | `/health` | Health check |
| GET | `/created` | Status code example |
| POST | `/echo` | Echo back message |

#### 💡 Try It

```bash
# Basic request
curl http://localhost:3000/

# Echo endpoint
curl -X POST -d "Hello Axum!" http://localhost:3000/echo
```

#### 📚 Key Concepts

##### Handler Functions
```rust
async fn hello() -> &'static str {
    "Hello, World!"
}
```

#### axum::serve (New in 0.8)
```rust
let listener = tokio::net::TcpListener::bind("0.0.0.0:3000").await?;
axum::serve(listener, app).await?;
```

#### ▶️ Next Module

Continue to [Module 02: Routing](../module-02-routing) to learn about path parameters and nested routes.

### Explain for some points 

`-> &'static str` means the function returns a string slice reference that's valid for the entire duration of the program.

**Why this works here:**

`"Hello, World! 🦀"` is a *string literal*. String literals in Rust aren't heap-allocated at runtime — they're baked directly into the compiled binary (in the read-only data section), and they exist for as long as the program runs. So the compiler automatically gives every string literal the `'static` lifetime. That's why you can return `&'static str` from a function with no arguments and no local data — you're just handing out a reference to data that lives forever, not to something on the stack that would disappear when the function returns.

**Why you need the lifetime annotation at all:**

Normally, if you return a `&str` (a reference) from a function, Rust needs to know how long that reference stays valid — that's what lifetimes are for. If you just wrote:

```rust
fn hello_world() -> &str {
    "Hello, World!"
}
```

Rust would actually accept this specific case too (it can infer `'static` from a literal), but in general, whenever a function returns a borrowed reference, Rust needs to tie that reference's lifetime to *something* — usually an input parameter's lifetime, or an explicit `'static`. Since `hello_world()` takes no arguments, there's nothing else to tie it to, so `'static` is the only lifetime that makes sense.

**Why axum specifically needs this:**

This isn't really an axum requirement — it's a general Rust rule. But axum handlers do need their return types to implement axum's `IntoResponse` trait, and `&'static str` implements `IntoResponse` (it gets converted into a `200 OK` response with the string as the body, `text/plain` content type). If you tried to return an owned `String` instead, that also implements `IntoResponse` — and is actually the more common/flexible choice when the string is built dynamically (e.g. `format!(...)`), since a `String` owns its data and doesn't need a lifetime annotation at all.

**Rule of thumb:**
- Returning a fixed, hardcoded string → `&'static str` works and is cheap (no allocation).
- Returning something built at runtime (formatted, from a database, etc.) → return `String` instead, since you can't get a `'static` reference to something you just created.